# DQL: SELECT, WHERE, GROUP BY, HAVING, ORDER BY

Data Query Language is used to read and shape data. This lesson covers the most common query clauses using the employee and department data.


## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## SELECT

Use `select` to choose columns. In PySpark you can use the DataFrame API or Spark SQL.

In plain English: `SELECT` means "show me these columns." It does not remove rows by itself; it only controls which columns appear in the final output.

Key points:

* `emp.select(...)` is the PySpark DataFrame version.
* `SELECT ... FROM emp` is the SQL version.
* Both examples return employee columns from the same `emp` data.


In [ ]:
emp.select("empno", "ename", "job", "sal", "deptno").show(10)

spark.sql("""
SELECT empno, ename, job, sal, deptno
FROM emp
LIMIT 10
""").show()


## WHERE

Use `where` or `filter` to keep only rows that match a condition.

In plain English: `WHERE` means "keep only the rows that pass this test." A condition is an expression that is either true or false for each row.

Key points:

* Rows with salary below the cutoff are removed from the result.
* The original DataFrame is not changed; Spark creates a new filtered result.
* `where` and `filter` mean the same thing in PySpark.


In [ ]:
emp.where(F.col("sal") >= 3000).select("empno", "ename", "job", "sal").show()

spark.sql("""
SELECT empno, ename, job, sal
FROM emp
WHERE sal >= 3000
""").show()


## GROUP BY

Use `groupBy` to collect rows into groups, then calculate summaries for each group.

In plain English: `GROUP BY` makes buckets of rows. After rows are grouped, aggregate functions such as `count`, `avg`, and `sum` calculate one answer per bucket.

Key points:

* Grouping by `deptno` creates one result row per department number.
* Every selected column must either be part of the group or be calculated with an aggregate function.
* This is how we answer questions like "how many employees are in each department?"


In [ ]:
emp.groupBy("deptno").agg(
    F.count("*").alias("employee_count"),
    F.round(F.avg("sal"), 2).alias("avg_salary"),
    F.sum("sal").alias("total_salary")
).orderBy("deptno").show()

spark.sql("""
SELECT deptno, COUNT(*) AS employee_count, ROUND(AVG(sal), 2) AS avg_salary, SUM(sal) AS total_salary
FROM emp
GROUP BY deptno
ORDER BY deptno
""").show()


## HAVING

`HAVING` filters grouped results. In the DataFrame API, aggregate first and then filter the summary rows.

In plain English: `HAVING` is like `WHERE`, but it filters after grouping has already happened.

Use this memory trick:

* `WHERE` filters individual rows before aggregation.
* `HAVING` filters grouped summary rows after aggregation.

In this example, Spark first totals salary by department and then keeps only departments whose total salary is high enough.


In [ ]:
dept_summary = emp.groupBy("deptno").agg(
    F.count("*").alias("employee_count"),
    F.sum("sal").alias("total_salary")
)

dept_summary.where(F.col("total_salary") > 15000).orderBy("total_salary", ascending=False).show()

spark.sql("""
SELECT deptno, COUNT(*) AS employee_count, SUM(sal) AS total_salary
FROM emp
GROUP BY deptno
HAVING SUM(sal) > 15000
ORDER BY total_salary DESC
""").show()


## ORDER BY

Use `orderBy` to sort the final result. Sorting is usually one of the last steps in a query.

In plain English: `ORDER BY` sorts the final answer so humans can read it more easily.

Key points:

* `desc()` means descending order, so larger salaries appear first.
* Sorting usually happens near the end of a query.
* If two rows have the same salary, the second sort column can make the order predictable.


In [ ]:
emp.select("empno", "ename", "job", "sal").orderBy(F.col("sal").desc(), F.col("ename")).show(10)

spark.sql("""
SELECT empno, ename, job, sal
FROM emp
ORDER BY sal DESC, ename
LIMIT 10
""").show()
